In [1]:
import numpy as np
import pandas as pd
import torch.nn as nn

from src.data import (ROOT, load_fd001, scale_features,
                      make_train_windows, make_test_windows)
from src.models import LSTMRegressor
from src.losses import msle, mape, tweedie, nasa_loss
from src.train import train_model, predict, rmse, nasa_score, set_seed

In [2]:
WINDOW = 30
train, test, test_rul, feats = load_fd001()
train, test, _ = scale_features(train, test, feats)

X_all, y_all, units = make_train_windows(train, feats, WINDOW)
X_test = make_test_windows(test, feats, WINDOW)
y_test = np.minimum(test_rul, 125)

# same validation split as notebook 03
rng = np.random.default_rng(42)
val_units = rng.choice(np.unique(units), size=20, replace=False)
val_mask = np.isin(units, val_units)
X_tr, y_tr = X_all[~val_mask], y_all[~val_mask]
X_val, y_val = X_all[val_mask], y_all[val_mask]
print(X_tr.shape, X_val.shape, X_test.shape)

(14070, 30, 17) (3661, 30, 17) (100, 30, 17)


In [3]:
LOSSES = {"MSE": nn.MSELoss(), "MAE": nn.L1Loss(), "MSLE": msle,
          "MAPE": mape, "Tweedie": tweedie, "NASA": nasa_loss}
SEEDS = [0, 1, 2]

rows = []
for name, loss_fn in LOSSES.items():
    for seed in SEEDS:
        set_seed(seed)    # import set_seed from src.train
        model = LSTMRegressor(n_features=len(feats), out_bias=float(y_tr.mean()))
        model, _ = train_model(model, X_tr, y_tr, X_val, y_val, loss_fn=loss_fn,
                               epochs=60, seed=seed, verbose=False)
        pred = np.clip(predict(model, X_test), 0, 125)   # same post-processing for every loss
        rows.append(dict(loss=name, seed=seed,
                         RMSE=rmse(y_test, pred),
                         MAE=float(np.mean(np.abs(y_test - pred))),
                         NASA=nasa_score(y_test, pred)))
        print(rows[-1])

results = pd.DataFrame(rows)
results.to_csv(ROOT / "results" / "loss_comparison.csv", index=False)   # ROOT is defined in src/data.py: from src.data import ROOT
print(results.groupby("loss")[["RMSE", "MAE", "NASA"]].agg(["mean", "std"]).round(2))

TypeError: train_model() got an unexpected keyword argument 'verbose'